**Molecular physical pharmacy, 3FC003**

**Molecular dynamics exercise**

# High-throughput peptide self-assembly

**Introduction**

This exercise will introduce you to combinatorial screening for peptide self-assembly using the Martini force field for proteins. The process is automated using a number of bash scripts, Gromacs tools and scripting capabilities within the visual molecular dynamics (VMD) program. After the simulations, visual inspection is done using VMD and analysis of the assembled structures is done using Gromacs tools.

NOTE that the exercise is written for versions 5.1 or 2016/2018 of Gromacs and will show errors when used with earlier versions

# Important:

Before start with this lab click in the edit menu and select clear all output

# 1. Setting up the computing environment

In [1]:
# Installing necessary packages
!pip install py3Dmol

In [2]:
# Cloning the course repository
!git clone https://github.com/computationalpharmaceutics/3FC003.git

Cloning into '3FC003'...
remote: Enumerating objects: 568, done.
remote: Total 568 (delta 0), reused 0 (delta 0), pack-reused 568 (from 2)
Receiving objects: 100% (568/568), 85.56 MiB | 29.19 MiB/s, done.
Resolving deltas: 100% (251/251), done.


In [3]:
# Installing GROMACS with CPU support only
!apt install gromacs &> /dev/null
!gmx -version

In [5]:
# Setting the path to GROMACS
import os
os.environ["PATH"] += ":/usr/local/gromacs/bin"

# Looking for the working directory
!pwd

/content


In [6]:
# Changing the working directory to

import os
os.chdir('/content/3FC003/HT_peptide_self_assembly')
!pwd

/content/3FC003/HT_peptide_self_assembly


# 1 Introduction

This exercise will introduce you to combinatorial screening for peptide self-assembly using the Martini force field for proteins. The process is automated using a number of bash scripts, Gromacs tools and scripting capabilities within the visual molecular dynamics (VMD) program. After the simulations, visual inspection is done using VMD and analysis of the assembled structures is done using Gromacs tools. NOTE that the exercise is written for versions 5.1 or 2016/2018 of Gromacs and will show errors when used with earlier versions.

- The exercise discusses the self-assembly of short peptides as an example system.
- The material for the exercise is located in the file **Peptide_assembly.tgz**.
- Unpack the directory tree (it expands to a directory called Peptide_assembly_GMX5-2016):

The material is organized in a directory tree that is numbered by the subsections of this tutorial:

 1_Background
  
 2_Creating_coordinates
  
 3_Coarse-graining

 4_Running_simulations

 5_Analysis

Each directory tree contains the files required for the tutorial. The results of a successful execution of the tutorial are also provided; this enables you to check your work and also to start anywhere if you run into some problem we cannot easily solve together. For example, if you want to jump in at step 4, you can enter the directory 3_Done and continue there in the directory 4_Running_simulations. Be advised that you do so at your own peril...

The main idea of this exercise is to show you, and practice, the commands to get through the different steps for a single peptide. However, in the files provided, you will also find several scripts that can help you perform the operations automatically, and set up a high throughput assay.

Finally, this exercise is based on a tutorial available on the martini website (cgmartini.nl), which is gratefully acknowledged.

**1. Background**

Molecular self-assembly of oligopeptides into nanostructures holds much promise for a range of potential applications in biomedicine, food science, cosmetics and nanotechnology. This class of materials is highly versatile because of the combinatorial complexity achieved by combining 20 amino acids into peptide building blocks with a wide range of chemical functionality. The use of very short peptides is especially attractive, enhancing opportunities for rational design combined with robustness, scalability and cost reduction.

Two main challenges are currently limiting the expansion of this field. Most examples of short peptides contain only hydrophobic amino acids. This is no surprise as hydrophobic interactions dominate self-assembly in water, but it also limits their aqueous solubility and restricts potential applications. Secondly, in spite of two decades of intensive research since the first examples of short self-assembling peptides, most examples have been either discovered by serendipity or by mapping onto known sequence design rules from biological systems. Using Martini, the self-assembly of thousands of peptides can be tested in silico, before spending time and resources in the lab. Experimental validation has shown that Martini not only accurately represents the level of aggregation between peptides, but also informs on the supramolecular structure of the nanosized assemblies.

In the field of peptide nanomaterials, it is common practice to ‘protect’ the N- and/or Cterminus of a peptide to introduce specific interactions and remove charge-charge repulsions. Examples include acetyl, Fmoc, naphthalene, pyrene and t-Boc functional groups at the N-terminus, or amide and ester groups at the C-terminus.

The directory 1_Background contains some references and further reading for those that are interested, but reading this is optional.

In this exercise we will apply a combinatorial screening protocol to tripeptides, specifically those discovered by Ray et al. They found that placing tyrosine residues at both the N- and the C-terminus of a tripeptide drives the system to crystallize into hollow nanotubes with a 5 Å inner diameter. They showed this works for Boc-Tyr-X-Tyr-OMe peptides, where X = Val, Ile, and that mutating either of the tyrosine residues prevents nanotube formation. This begs the question if peptides with other middle residues will maintain the nanotube conformation or will change their morphology. Additionally, the nanotubes were created by crystallization from a water/methanol mixture, while for realistic applications, the stability of the nanotubes in an aqueous environment should be tested.

**Creating peptide coordinates**

To simplify the tripeptides for this exercise, we will treat Boc-Tyr-X-Tyr-OMe as a simple TyrX-Tyr peptide with uncharged termini, so that all parameters for coarse-graining are available within the Martini force field for proteins. We want to test all 20 mutations of the middle amino acid (X). An easy way to create the 20 different peptide coordinates is using a molecular builder such as Avogadro, VMD or Pymol.

## 2. Creating the 20 Tyr–X–Tyr coordinates with PyMOL

The supplied `make_peptides.sh` is shown below for reference. It uses VMD's `psfgen` plugin, so merely changing `vmd='vmd'` is not sufficient on a runtime where VMD is absent. The notebook-native workflow below uses PyMOL's peptide builder to generate the same 20 sequence variants and therefore does not require a VMD executable.

Files are written as `TYR-XXX-TYR_aa.pdb`, matching the names expected by `3_Coarse-graining/setup_tripeptides.sh`. PyMOL completes the terminal groups and adds hydrogens to give neutral NH₂/COOH atomistic structures. The supplied coarse-graining script also passes `-nt` to `martinize.py` so the Martini termini remain neutral.

In [ ]:
# Install the open-source PyMOL Python package for building and rendering.
%pip install -q pymol-open-source

In [ ]:
# Locate 2_Creating_coordinates in Colab or in a local checkout.
from pathlib import Path
import os
import stat

candidate_dirs = [
    Path('/content/3FC003/Peptide_assembly_GMX5-2016/2_Creating_coordinates'),
    Path.cwd() / 'Peptide_assembly_GMX5-2016/2_Creating_coordinates',
    Path.cwd().parent / 'Peptide_assembly_GMX5-2016/2_Creating_coordinates',
    Path.cwd(),
]
COORD_DIR = next(
    (path.resolve() for path in candidate_dirs if (path / 'make_peptides.sh').is_file()),
    None,
)
if COORD_DIR is None:
    raise FileNotFoundError(
        'Could not find 2_Creating_coordinates. Run the repository clone/setup cells first.'
    )

os.chdir(COORD_DIR)
make_script = COORD_DIR / 'make_peptides.sh'
make_script.chmod(
    make_script.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
)

print(f'Working directory: {COORD_DIR}')
print(f'make_peptides.sh executable: {os.access(make_script, os.X_OK)}')
print('\n--- supplied make_peptides.sh (VMD workflow; inspected only) ---')
print(make_script.read_text())

coarse_grain_script = COORD_DIR.parent / '3_Coarse-graining/setup_tripeptides.sh'
if coarse_grain_script.is_file():
    assert '-nt' in coarse_grain_script.read_text(), (
        'The coarse-graining command must include -nt for neutral termini.'
    )
    print('Verified: setup_tripeptides.sh uses martinize.py -nt (neutral termini).')

In [ ]:
# Generate all 20 Tyr-X-Tyr all-atom coordinate files with PyMOL.
# pymol2 creates an isolated headless session and works on Linux/macOS.
import pymol2

pymol_session = pymol2.PyMOL()
pymol_session.start()
cmd = pymol_session.cmd

AA3_TO_AA1 = {
    'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
    'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
    'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
    'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y',
}

cmd.reinitialize()
created_files = []
for middle_3, middle_1 in AA3_TO_AA1.items():
    object_name = f'peptide_{middle_3}'
    sequence = f'Y{middle_1}Y'
    output_file = COORD_DIR / f'TYR-{middle_3}-TYR_aa.pdb'

    # ss=2 creates an extended beta-like starting conformation.
    cmd.fab(sequence, object_name, resi=1, chain='A', segi='PEP', ss=2)

    # PyMOL's fab builder omits the second C-terminal oxygen; add it as OXT.
    cmd.edit(f'{object_name} and resi 3 and name C')
    cmd.attach('O', 1, 1)
    assert cmd.count_atoms(f'{object_name} and resi 3 and name O01') == 1
    cmd.alter(f'{object_name} and resi 3 and name O01', "name='OXT'")
    cmd.h_add(object_name)  # Complete neutral NH2 and COOH terminal groups.
    cmd.sort(object_name)
    cmd.save(str(output_file), object_name, state=1, format='pdb')
    cmd.delete(object_name)
    created_files.append(output_file)

print(f'Created {len(created_files)} PDB files in {COORD_DIR}')

In [ ]:
# Confirm that every output contains exactly the expected three residues.
def pdb_residues_and_atom_count(pdb_file):
    residues = []
    atom_count = 0
    for line in pdb_file.read_text().splitlines():
        if line.startswith(('ATOM  ', 'HETATM')):
            atom_count += 1
            if line[12:16].strip() == 'CA':
                residues.append(line[17:20].strip())
    return residues, atom_count

generated = sorted(COORD_DIR.glob('TYR-*-TYR_aa.pdb'))
assert len(generated) == 20, f'Expected 20 PDB files, found {len(generated)}.'

validation_rows = []
for pdb_file in generated:
    middle = pdb_file.name.split('-')[1]
    residues, atom_count = pdb_residues_and_atom_count(pdb_file)
    expected = ['TYR', middle, 'TYR']
    assert residues == expected, f'{pdb_file.name}: expected {expected}, found {residues}'
    pdb_lines = pdb_file.read_text().splitlines()
    assert any(
        line.startswith('ATOM  ') and line[12:16].strip() == 'OXT'
        for line in pdb_lines
    ), f'{pdb_file.name}: missing the C-terminal OXT atom'
    assert any(
        line.startswith('ATOM  ') and line[76:78].strip() == 'H'
        for line in pdb_lines
    ), f'{pdb_file.name}: missing all-atom hydrogens'
    validation_rows.append((pdb_file.name, '-'.join(residues), atom_count))

print(f'{"file":28s}  {"sequence":11s}  atoms')
print('-' * 50)
for file_name, sequence, atom_count in validation_rows:
    print(f'{file_name:28s}  {sequence:11s}  {atom_count:5d}')
print('\nValidation passed for all 20 Tyr-X-Tyr structures.')

### Visualize one generated peptide with PyMOL

Change `PEPTIDE_TO_VIEW` to any generated name (for example, `TYR-ILE-TYR` or `TYR-TRP-TYR`). PyMOL renders the structure headlessly and the resulting PNG is displayed in the notebook.

In [ ]:
# Render a generated peptide using PyMOL (not VMD).
from IPython.display import Image, display

PEPTIDE_TO_VIEW = 'TYR-VAL-TYR'
pdb_to_view = COORD_DIR / f'{PEPTIDE_TO_VIEW}_aa.pdb'
if not pdb_to_view.is_file():
    raise FileNotFoundError(f'Run the generation cell first: {pdb_to_view}')

cmd.reinitialize()
cmd.load(str(pdb_to_view), 'peptide')
cmd.hide('everything', 'all')
cmd.show('sticks', 'peptide')
cmd.color('gray70', 'peptide and elem C')
cmd.color('blue', 'peptide and elem N')
cmd.color('red', 'peptide and elem O')
cmd.color('yellow', 'peptide and elem S')
cmd.set('stick_radius', 0.18)
cmd.set('ray_opaque_background', 0)
cmd.bg_color('white')
cmd.orient('peptide')
cmd.zoom('peptide', buffer=2.0)

render_file = COORD_DIR / f'{PEPTIDE_TO_VIEW}_PyMOL.png'
cmd.png(str(render_file), width=1000, height=650, dpi=150, ray=1)
display(Image(filename=str(render_file)))
print(f'PyMOL render saved to: {render_file}')